In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseNetwork(nn.Module) :
    def __init__(self, embedding_size=128) :
        super(SiameseNetwork, self).__init__()

        self.embedding_size = embedding_size

        self.convnet = nn.Sequential (
            nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 7, stride = 1, padding = 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size = 5, stride = 1, padding = 2),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(512, embedding_size)

    def forward(self, x) :
        x = self.convnet(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = F.normalize(x, p = 2, dim = 1)
        return x

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SiameseNetwork(embedding_size=128).to(device)
model.load_state_dict(torch.load("face_recognition_model.pth", map_location=device))
model.eval()


SiameseNetwork(
  (convnet): Sequential(
    (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (fc): Linear(in_features=512, out_features=128, bias=True)
)

In [ ]:

from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

# Load images
img1 = transform(Image.open("face_recognition/wanted_people/2.jpg").convert("RGB")).unsqueeze(0).to(device)
img2 = transform(Image.open("face_recognition/corrent_person/2.jpg").convert("RGB")).unsqueeze(0).to(device)

with torch.no_grad():
    emb1 = model(img1)
    emb2 = model(img2)

distance = torch.nn.functional.pairwise_distance(emb1, emb2)
print("Distance:", distance.item())



Distance: 0.8864117860794067
Different people
